In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [5]:
shamem=r"""

#include <cstdio>
#include <cuda_runtime.h>

__global__ void reduce_sum(const float* in, float* blockSums, int n){
    extern __shared__ float sdata[];

    int tid=threadIdx.x;
    int i=blockIdx.x*blockDim.x+tid;

    sdata[tid]=(i<n)?in[i]:0.0f;

    __syncthreads();

    for(int s=blockDim.x/2; s>0; s=s/2){
        if(tid<s){
            sdata[tid]=sdata[tid]+sdata[tid+s];
        }
        __syncthreads();
    }

    if(tid==0){
        blockSums[blockIdx.x]=sdata[0];
    }
}

int main(){
    const int n=1<<20;
    const int threads=256;
    const int blocks=(n+threads-1)/threads;

    float *in, *blockSums;
    cudaMallocManaged(&in, n*sizeof(float));
    cudaMallocManaged(&blockSums, blocks*sizeof(float));

    for(int i=0; i<n; i++) in[i]=1.0f;

    reduce_sum<<<blocks, threads, threads*sizeof(float)>>>(in, blockSums, n);
    cudaDeviceSynchronize();

    float total=0;
    for(int b=0; b<blocks; b++) total+=blockSums[b];
    
    printf("Sum: %f and expexted %d", total, n);

    cudaFree(in);
    cudaFree(blockSums);

    return 0;
}

"""

with open('shared_memory.cu', 'w') as f:
    f.write(shamem)

In [6]:
!nvcc -arch=sm_75 shared_memory.cu -o shamem

In [7]:
!./shamem

Sum: 1048576.000000 and expexted 1048576